# Silver Layer: Booking Table Cleaning & Validation

Cleanses `skyledger_airlines.bronze.booking` into `skyledger_airlines.silver.booking`.
Applies primary key validation (§3.1), date range validation (§3.3), and referential integrity checks (§3.4).
Invalid records are quarantined in `silver.booking_quarantine` with comma-separated rejection reasons.
Includes Bronze-Silver-Quarantine reconciliation and a data-quality report.

**Relationships:** `passenger` → `booking` → `flight_booking` (core booking fact chain).
FK `bk_passengerid` validated against `silver.passenger` (parent-before-child order).

**Date rule (§3.3):** `bk_bookingdate` and `bk_flightdate` must fall within 1994-01-01 to 1998-12-31 inclusive.

In [0]:
%sql
select * from skyledger_airlines.bronze.booking

In [0]:
# ── Step 1: Read Bronze, Cast Types, Drop Metadata Columns ──
from pyspark.sql.functions import col, when, lit, count as _count, concat_ws, to_date
from pyspark.sql.window import Window

bronze_booking = spark.table("skyledger_airlines.bronze.booking")

# Drop metadata columns: _ingestion_timestamp, _source_file, _pipeline_run_id
# Cast key columns to INT, price to DOUBLE, dates to DATE
booking_typed = bronze_booking.select(
    col("bk_bookingid").cast("int").alias("bk_bookingid"),
    col("bk_passengerid").cast("int").alias("bk_passengerid"),
    col("bk_status").cast("string").alias("bk_status"),
    col("bk_totalprice").cast("double").alias("bk_totalprice"),
    col("bk_bookingdate").cast("date").alias("bk_bookingdate"),
    col("bk_flightdate").cast("date").alias("bk_flightdate"),
    col("bk_bookingpriority").cast("string").alias("bk_bookingpriority"),
    col("bk_agent").cast("string").alias("bk_agent"),
    col("bk_comment").cast("string").alias("bk_comment"),
)

bronze_count = booking_typed.count()
print(f"Bronze booking rows read: {bronze_count}")
display(booking_typed.limit(10))

In [0]:
# ── Step 2: Primary Key Validation (§3.1) ──
# Rule: bk_bookingid must be NOT NULL and UNIQUE.
# Failure codes: NULL_PK, DUPLICATE_PK

booking_flagged = booking_typed.withColumn(
    "is_null_pk",
    when(col("bk_bookingid").isNull(), lit(True)).otherwise(lit(False)),
)

# Window-based duplicate detection
pk_window = Window.partitionBy("bk_bookingid")
booking_flagged = booking_flagged.withColumn(
    "_pk_count", _count("*").over(pk_window)
).withColumn(
    "is_duplicate_pk",
    when((col("bk_bookingid").isNotNull()) & (col("_pk_count") > 1), lit(True)).otherwise(lit(False)),
)

null_pk_count = booking_flagged.filter(col("is_null_pk")).count()
duplicate_pk_count = booking_flagged.filter(col("is_duplicate_pk")).count()

print(f"NULL_PK failures:      {null_pk_count}")
print(f"DUPLICATE_PK failures: {duplicate_pk_count}")

In [0]:
# ── Step 3: Numeric Range & Date Range Validation (§3.2, §3.3) ──
# Rule: bk_totalprice must be strictly greater than zero.
# Rule: bk_bookingdate and bk_flightdate must fall within 1994-01-01 to 1998-12-31 inclusive.
# Failure codes: INVALID_TOTALPRICE, INVALID_DATE_BOOKINGDATE, INVALID_DATE_FLIGHTDATE

# Numeric range check — bk_totalprice must be > 0
booking_flagged = booking_flagged.withColumn(
    "is_invalid_totalprice",
    when(col("bk_totalprice").isNull() | (col("bk_totalprice") <= 0), lit(True)).otherwise(lit(False)),
)

# Date range checks
date_min = "1994-01-01"
date_max = "1998-12-31"

booking_flagged = booking_flagged.withColumn(
    "is_invalid_bookingdate",
    when(
        col("bk_bookingdate").isNull()
        | (col("bk_bookingdate") < lit(date_min))
        | (col("bk_bookingdate") > lit(date_max)),
        lit(True),
    ).otherwise(lit(False)),
).withColumn(
    "is_invalid_flightdate",
    when(
        col("bk_flightdate").isNull()
        | (col("bk_flightdate") < lit(date_min))
        | (col("bk_flightdate") > lit(date_max)),
        lit(True),
    ).otherwise(lit(False)),
)

invalid_bookingdate_count = booking_flagged.filter(col("is_invalid_bookingdate")).count()
invalid_flightdate_count = booking_flagged.filter(col("is_invalid_flightdate")).count()

invalid_totalprice_count = booking_flagged.filter(col("is_invalid_totalprice")).count()
print(f"INVALID_TOTALPRICE failures:       {invalid_totalprice_count}")
print(f"INVALID_DATE_BOOKINGDATE failures: {invalid_bookingdate_count}")
print(f"INVALID_DATE_FLIGHTDATE failures:  {invalid_flightdate_count}")

In [0]:
# ── Step 4: Referential Integrity Validation (§3.4) ──
# Rule: bk_passengerid must resolve to a valid pa_passengerid in silver.passenger.
# Failure code: INVALID_FK_PASSENGER
#
# Parent-before-child order: passenger is the parent of booking.
# Uses silver.passenger (already validated) as the reference.

valid_passenger_ids = (
    spark.table("skyledger_airlines.silver.passenger")
    .select(col("pa_passengerid").cast("int").alias("pa_passengerid"))
    .filter(col("pa_passengerid").isNotNull())
    .distinct()
)

# Left join: rows with no match (or null bk_passengerid) get pa_passengerid = null → invalid FK
booking_flagged = booking_flagged.join(
    valid_passenger_ids,
    booking_flagged["bk_passengerid"] == valid_passenger_ids["pa_passengerid"],
    "left",
).withColumn(
    "is_invalid_fk_passenger",
    when(col("pa_passengerid").isNull(), lit(True)).otherwise(lit(False)),
)

invalid_fk_count = booking_flagged.filter(col("is_invalid_fk_passenger")).count()
print(f"INVALID_FK_PASSENGER failures: {invalid_fk_count}")

In [0]:
# ── Step 5: Build Rejection Reasons, Split Clean vs Quarantine, Write Tables ──

# Build comma-separated rejection_reason from all flag columns
booking_flagged = booking_flagged.withColumn(
    "rejection_reason",
    concat_ws(
        ", ",
        when(col("is_null_pk"), lit("NULL_PK")),
        when(col("is_duplicate_pk"), lit("DUPLICATE_PK")),
        when(col("is_invalid_totalprice"), lit("INVALID_TOTALPRICE")),
        when(col("is_invalid_bookingdate"), lit("INVALID_DATE_BOOKINGDATE")),
        when(col("is_invalid_flightdate"), lit("INVALID_DATE_FLIGHTDATE")),
        when(col("is_invalid_fk_passenger"), lit("INVALID_FK_PASSENGER")),
    ),
)

# Clean records: passed every check (rejection_reason is empty string)
silver_booking = booking_flagged.filter(col("rejection_reason") == "").select(
    "bk_bookingid", "bk_passengerid", "bk_status", "bk_totalprice",
    "bk_bookingdate", "bk_flightdate", "bk_bookingpriority", "bk_agent", "bk_comment",
)

# Quarantined records: failed one or more checks
quarantine_booking = booking_flagged.filter(col("rejection_reason") != "").select(
    "bk_bookingid", "bk_passengerid", "bk_status", "bk_totalprice",
    "bk_bookingdate", "bk_flightdate", "bk_bookingpriority", "bk_agent", "bk_comment",
    "rejection_reason",
)

silver_count = silver_booking.count()
quarantine_count = quarantine_booking.count()

print(f"Silver rows:     {silver_count}")
print(f"Quarantine rows: {quarantine_count}")

# Write Silver table (overwrite for idempotency — same result on every run)
(silver_booking.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.booking"))

# Write Quarantine table (overwrite for idempotency)
(quarantine_booking.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.booking_quarantine"))

print("✅ Tables written: silver.booking, silver.booking_quarantine")

In [0]:
# ── Step 6: Bronze-Silver-Quarantine Reconciliation (§3.5) ──
# Bronze row count must equal Silver + Quarantine for the booking table.

bronze_total = spark.table("skyledger_airlines.bronze.booking").count()
silver_total = spark.table("skyledger_airlines.silver.booking").count()
quarantine_total = spark.table("skyledger_airlines.silver.booking_quarantine").count()

print(f"Bronze row count:      {bronze_total}")
print(f"Silver row count:      {silver_total}")
print(f"Quarantine row count:  {quarantine_total}")
print(f"Silver + Quarantine:   {silver_total + quarantine_total}")

assert bronze_total == silver_total + quarantine_total, (
    f"RECONCILIATION FAILED: Bronze ({bronze_total}) != "
    f"Silver ({silver_total}) + Quarantine ({quarantine_total})"
)

print("✅ Reconciliation PASSED: Bronze = Silver + Quarantine")

In [0]:
# ── Step 7: Data Quality Report ──
# Documents every check executed, even when zero rows fail.

dq_report = spark.createDataFrame(
    [
        ("booking", "NULL_PK", "bk_bookingid must not be null", null_pk_count, "Quarantine"),
        ("booking", "DUPLICATE_PK", "bk_bookingid must be unique across all rows", duplicate_pk_count, "Quarantine"),
        ("booking", "INVALID_TOTALPRICE", "bk_totalprice must be strictly greater than zero", invalid_totalprice_count, "Quarantine"),
        ("booking", "INVALID_DATE_BOOKINGDATE", "bk_bookingdate must fall within 1994-01-01 to 1998-12-31", invalid_bookingdate_count, "Quarantine"),
        ("booking", "INVALID_DATE_FLIGHTDATE", "bk_flightdate must fall within 1994-01-01 to 1998-12-31", invalid_flightdate_count, "Quarantine"),
        ("booking", "INVALID_FK_PASSENGER", "bk_passengerid must reference a valid pa_passengerid in silver.passenger", invalid_fk_count, "Quarantine"),
    ],
    ["table", "rule_code", "check_description", "failing_rows", "action_taken"],
)

display(dq_report)

# Show quarantined records if any exist
if quarantine_count > 0:
    print(f"\n── Quarantined Records ({quarantine_count} rows) ──")
    display(spark.table("skyledger_airlines.silver.booking_quarantine"))
else:
    print("\nNo quarantined records — all booking rows passed every check.")

# Show final Silver table
print(f"\n── Silver Booking Table ({silver_count} rows) ──")
display(spark.table("skyledger_airlines.silver.booking"))